# SegFormer Road Segmentation — Kaggle P100 / Google Colab

Trains **SegFormer** (Mix Transformer encoder + lightweight MLP decoder) on the DeepGlobe satellite road dataset.

SegFormer outperforms DeepLabV3-ResNet101 while being **smaller than ResNet50**:

| Model | Params | Road IoU (approx) | Notes |
|---|---|---|---|
| DeepLabV3-ResNet50 | 42 M | ~61–64 % | baseline |
| DeepLabV3-ResNet101 | 61 M | ~63–65 % | +1–2 pts |
| **SegFormer-B2** | **25 M** | **~67–70 %** | best size/accuracy |
| **SegFormer-B5** | **82 M** | **~70–73 %** | highest accuracy |

Key differences from the ResNet notebook:
- **AdamW + linear warmup + cosine decay** (transformers need this, not ReduceLROnPlateau)
- **Lower LR**: 6e-5 (transformers are sensitive to high learning rates)
- **`build_segformer(variant)`** instead of `build_model(backbone)`
- Everything else — dataset, loss, metrics, checkpointing — is identical

**Resuming after disconnect:** re-run all cells — checkpoint reloads automatically.

## Cell 1 — Platform

In [ ]:
import os

# ── Change to "colab" when running on Google Colab ────────────────────────────
PLATFORM = "kaggle"

if PLATFORM == "kaggle":
    OUTPUT_DIR      = "/kaggle/working"
    CHECKPOINT_PATH = "/kaggle/working/segformer_checkpoint.pth"
    FINAL_MODEL     = "/kaggle/working/segformer_road_final.pth"
    BEST_MODEL      = "/kaggle/working/segformer_best.pth"
else:
    OUTPUT_DIR      = "/content"
    CHECKPOINT_PATH = "/content/segformer_checkpoint.pth"
    FINAL_MODEL     = "/content/segformer_road_final.pth"
    BEST_MODEL      = "/content/segformer_best.pth"

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Platform : {PLATFORM}")
print(f"Output   : {OUTPUT_DIR}")

## Cell 2 — Install Dependencies

In [ ]:
%pip install kagglehub torch torchvision transformers tqdm opencv-python scikit-image -q
print("✅ Dependencies installed")

## Cell 3 — GPU Check

In [ ]:
import torch

print(f"PyTorch    : {torch.__version__}")
print(f"CUDA       : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU        : {name}  ({vram:.1f} GB VRAM)")
    if "P100" in name:
        print("✅ Kaggle P100 — B2 batch=8 or B5 batch=4 at 512px")
    elif "T4" in name:
        print("✅ Colab T4 — B2 batch=4 at 512px recommended")
    elif "A100" in name:
        print("✅ A100 — B5 batch=16 at 512px or B2 batch=8 at 1024px")
else:
    raise RuntimeError("❌ No GPU found. Enable GPU in accelerator settings.")

## Cell 4 — Dataset Download

In [ ]:
import kagglehub

path = kagglehub.dataset_download("balraj98/deepglobe-road-extraction-dataset")
train_dir = os.path.join(path, "train")
if not os.path.isdir(train_dir):
    train_dir = path

sat_files = [f for f in os.listdir(train_dir) if f.endswith("_sat.jpg")]
print(f"Dataset    : {path}")
print(f"Train dir  : {train_dir}")
print(f"Images     : {len(sat_files)}")
assert len(sat_files) > 0, "No *_sat.jpg images found"

## Cell 5 — Hyperparameters

**Variant guide for P100 16 GB at 512px:**

| Variant | Params | Recommended batch | Est. time/epoch |
|---|---|---|---|
| b2 | 25 M | 8 | ~12–16 min |
| b5 | 82 M | 4 | ~20–28 min |

Start with **b2** to establish a baseline quickly, then run b5 for the final model.

In [ ]:
# ── Hyperparameters ───────────────────────────────────────────────────────────
VARIANT     = "b2"   # "b0" "b1" "b2" "b3" "b4" "b5"
IMG_SIZE    = 512    # SegFormer works best at 512px (its decoder upsamples 4×)
BATCH_SIZE  = 8      # P100/b2: 8.  P100/b5: 4.  T4/b2: 4.
NUM_EPOCHS  = 15     # Transformers converge slower than CNNs; 15 beats 10
LR          = 6e-5   # Transformers need lower LR than CNNs (1e-4 is too high)
WEIGHT_DECAY = 0.01  # AdamW weight decay — important for transformer regularisation
WARMUP_FRAC  = 0.1   # Fraction of total steps used for linear LR warmup
VAL_SPLIT   = 0.2
SEED        = 42
NUM_WORKERS = 4
USE_CLAHE   = True
USE_AMP     = torch.cuda.is_available()
DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")

total_train = int(len(sat_files) * (1 - VAL_SPLIT))
steps_per_epoch = total_train // BATCH_SIZE
total_steps     = steps_per_epoch * NUM_EPOCHS
warmup_steps    = int(total_steps * WARMUP_FRAC)

print(f"Variant    : SegFormer-{VARIANT.upper()}")
print(f"IMG_SIZE   : {IMG_SIZE}  BATCH={BATCH_SIZE}  EPOCHS={NUM_EPOCHS}")
print(f"LR         : {LR}  WD={WEIGHT_DECAY}  Warmup steps={warmup_steps}/{total_steps}")
print(f"AMP        : {USE_AMP}  Device: {DEVICE}")

## Cell 6 — Dataset Class + Train/Val Split

In [ ]:
import cv2
import random
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torchvision.transforms.functional as TF


def apply_clahe(image, clip_limit=2.0, tile_grid_size=(8, 8)):
    img_np = np.array(image)
    lab = cv2.cvtColor(img_np, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid_size)
    lab_enhanced = cv2.merge((clahe.apply(l), a, b))
    return Image.fromarray(cv2.cvtColor(lab_enhanced, cv2.COLOR_LAB2RGB))


class RoadSegmentationDataset(Dataset):
    def __init__(self, data_dir, file_list, img_size=512, augment=True, use_clahe=True):
        self.data_dir  = data_dir
        self.files     = sorted(file_list)
        self.augment   = augment
        self.use_clahe = use_clahe
        self.resize    = transforms.Resize((img_size, img_size))
        self.to_tensor = transforms.ToTensor()
        self.normalize = transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std =[0.229, 0.224, 0.225],
        )

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        name  = self.files[idx]
        image = Image.open(os.path.join(self.data_dir, name)).convert("RGB")
        mask  = Image.open(os.path.join(self.data_dir, name.replace("_sat.jpg", "_mask.png"))).convert("L")

        image = self.resize(image)
        mask  = self.resize(mask)

        if self.use_clahe:
            image = apply_clahe(image)

        if self.augment:
            if random.random() > 0.5:
                image, mask = TF.hflip(image), TF.hflip(mask)
            if random.random() > 0.5:
                image, mask = TF.vflip(image), TF.vflip(mask)
            if random.random() > 0.5:
                angle = random.choice([90, 180, 270])
                image, mask = TF.rotate(image, angle), TF.rotate(mask, angle)

        image = self.normalize(self.to_tensor(image))
        mask  = (self.to_tensor(mask) > 0.5).long().squeeze(0)
        return image, mask


# Deterministic 80/20 split
all_files = sorted(f for f in os.listdir(train_dir) if f.endswith("_sat.jpg"))
rng = random.Random(SEED)
shuffled = all_files[:]
rng.shuffle(shuffled)
split       = int(len(shuffled) * (1 - VAL_SPLIT))
train_files = shuffled[:split]
val_files   = shuffled[split:]

train_ds = RoadSegmentationDataset(train_dir, train_files, IMG_SIZE, augment=True,  use_clahe=USE_CLAHE)
val_ds   = RoadSegmentationDataset(train_dir, val_files,   IMG_SIZE, augment=False, use_clahe=USE_CLAHE)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

print(f"Train: {len(train_ds)} images  ({len(train_loader)} batches)")
print(f"Val  : {len(val_ds)} images  ({len(val_loader)} batches)")

## Cell 7 — Build SegFormer Model

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
from transformers import SegformerForSemanticSegmentation


class SegFormerWrapper(nn.Module):
    """
    Wraps HuggingFace SegFormer to match DeepLabV3's model(x) → {'out': tensor} interface.
    SegFormer's head outputs at H/4 × W/4; we upsample back to input resolution.
    """
    def __init__(self, hf_model):
        super().__init__()
        self.model = hf_model

    def forward(self, x):
        h, w   = x.shape[-2:]
        logits = self.model(pixel_values=x).logits          # (B, C, H/4, W/4)
        logits = F.interpolate(logits, size=(h, w),
                               mode="bilinear", align_corners=False)
        return {"out": logits}


model_id = f"nvidia/mit-{VARIANT}"
print(f"Downloading {model_id} weights from HuggingFace...")

hf_model = SegformerForSemanticSegmentation.from_pretrained(
    model_id,
    num_labels=2,                  # road / background
    ignore_mismatched_sizes=True,  # replaces pretrained 150-class head with new 2-class head
)
model = SegFormerWrapper(hf_model).to(DEVICE)

total_params  = sum(p.numel() for p in model.parameters()) / 1e6
train_params  = sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6
print(f"✅ SegFormer-{VARIANT.upper()} loaded  ({total_params:.1f} M params, {train_params:.1f} M trainable)")

## Cell 8 — Training Loop

Uses **AdamW + linear warmup + cosine decay** — the standard transformer recipe.  
ReduceLROnPlateau (used for ResNets) performs worse for transformers because they need the gradual warmup phase and a predictable decay curve.

In [ ]:
import math
from torch.optim import AdamW
from tqdm import tqdm


# ── Loss ──────────────────────────────────────────────────────────────────────
class DiceLoss(nn.Module):
    def __init__(self, smooth=1e-6):
        super().__init__()
        self.smooth = smooth

    def forward(self, predict, target):
        predict = torch.softmax(predict, dim=1)[:, 1, :, :]
        target  = target.float()
        inter   = (predict * target).sum(dim=(1, 2))
        union   = predict.sum(dim=(1, 2)) + target.sum(dim=(1, 2))
        return 1 - ((2. * inter + self.smooth) / (union + self.smooth)).mean()


class_weights = torch.tensor([1.0, 10.0]).to(DEVICE)
ce_loss       = nn.CrossEntropyLoss(weight=class_weights)
dice_loss     = DiceLoss()


def criterion(out, tgt):
    return ce_loss(out, tgt) + dice_loss(out, tgt)


# ── Optimizer: AdamW with separate LR for encoder vs decoder ──────────────────
# Transformers benefit from a lower LR on the pretrained encoder
encoder_params = [p for n, p in model.named_parameters() if "decode_head" not in n]
decoder_params = [p for n, p in model.named_parameters() if "decode_head"     in n]

optimizer = AdamW([
    {"params": encoder_params, "lr": LR},
    {"params": decoder_params, "lr": LR * 10},   # decoder head can learn faster
], weight_decay=WEIGHT_DECAY)


# ── LR schedule: linear warmup → cosine decay ─────────────────────────────────
def lr_lambda(current_step):
    if current_step < warmup_steps:
        return current_step / max(1, warmup_steps)   # linear warmup
    progress = (current_step - warmup_steps) / max(1, total_steps - warmup_steps)
    return max(0.0, 0.5 * (1.0 + math.cos(math.pi * progress)))  # cosine decay

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
scaler    = torch.cuda.amp.GradScaler(enabled=USE_AMP)
global_step = 0


# ── Checkpoint helpers ─────────────────────────────────────────────────────────
def save_checkpoint(epoch, model, optimizer, scheduler, scaler, val_iou, path):
    torch.save({
        "epoch":                epoch,
        "global_step":          global_step,
        "model_state_dict":     model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "scaler_state_dict":    scaler.state_dict(),
        "val_iou":              val_iou,
        "variant":              VARIANT,
        "img_size":             IMG_SIZE,
    }, path)


def load_checkpoint(path, model, optimizer, scheduler, scaler):
    global global_step
    if not os.path.exists(path):
        return 0, 0.0
    ckpt = torch.load(path, map_location=DEVICE)
    model.load_state_dict(ckpt["model_state_dict"])
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    scheduler.load_state_dict(ckpt["scheduler_state_dict"])
    if "scaler_state_dict" in ckpt:
        scaler.load_state_dict(ckpt["scaler_state_dict"])
    global_step = ckpt.get("global_step", 0)
    start    = ckpt["epoch"] + 1
    best_iou = ckpt.get("val_iou", 0.0)
    print(f"✅ Resumed from epoch {start}  val IoU={best_iou:.4f}  step={global_step}")
    return start, best_iou


# ── Validation ────────────────────────────────────────────────────────────────
def validate_epoch(model, loader):
    model.eval()
    ious, f1s = [], []
    with torch.no_grad():
        for imgs, masks in loader:
            imgs = imgs.to(DEVICE)
            with torch.cuda.amp.autocast(enabled=USE_AMP):
                out = model(imgs)["out"]
            preds = torch.argmax(out, dim=1).cpu()
            for i in range(preds.size(0)):
                p = preds[i].view(-1)
                t = masks[i].view(-1)
                tp = ((p == 1) & (t == 1)).sum().item()
                fp = ((p == 1) & (t == 0)).sum().item()
                fn = ((p == 0) & (t == 1)).sum().item()
                ious.append(tp / (tp + fp + fn + 1e-7))
                prec = tp / (tp + fp + 1e-7)
                rec  = tp / (tp + fn + 1e-7)
                f1s.append(2 * prec * rec / (prec + rec + 1e-7))
    model.train()
    return float(sum(ious) / len(ious)), float(sum(f1s) / len(f1s))


# ── Training loop ──────────────────────────────────────────────────────────────
start_epoch, best_iou = load_checkpoint(CHECKPOINT_PATH, model, optimizer, scheduler, scaler)

print(f"\n🚀 SegFormer-{VARIANT.upper()} on {DEVICE}  |  AMP={USE_AMP}  |  batch={BATCH_SIZE}  |  img={IMG_SIZE}px")
print(f"   Epochs {start_epoch + 1} → {NUM_EPOCHS}  |  Warmup={warmup_steps} steps")
print("=" * 70)

model.train()
val_iou = best_iou

for epoch in range(start_epoch, NUM_EPOCHS):
    epoch_loss = 0.0
    bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}", unit="batch")

    for i, (imgs, masks) in enumerate(bar):
        imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
        optimizer.zero_grad()

        with torch.cuda.amp.autocast(enabled=USE_AMP):
            out  = model(imgs)["out"]
            loss = criterion(out, masks)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        global_step += 1

        epoch_loss += loss.item()
        lr_now = optimizer.param_groups[0]["lr"]
        bar.set_postfix(loss=f"{loss.item():.4f}", avg=f"{epoch_loss/(i+1):.4f}", lr=f"{lr_now:.1e}")

    avg_loss         = epoch_loss / len(train_loader)
    val_iou, val_f1  = validate_epoch(model, val_loader)
    lr_now           = optimizer.param_groups[0]["lr"]

    print(f"Epoch {epoch+1:>2}/{NUM_EPOCHS}  "
          f"loss={avg_loss:.4f}  val_iou={val_iou:.4f}  val_f1={val_f1:.4f}  lr={lr_now:.2e}")

    save_checkpoint(epoch, model, optimizer, scheduler, scaler, val_iou, CHECKPOINT_PATH)

    if val_iou > best_iou:
        best_iou = val_iou
        torch.save({
            "model_state_dict": model.state_dict(),
            "epoch":    epoch,
            "val_iou":  val_iou,
            "variant":  VARIANT,
            "img_size": IMG_SIZE,
        }, BEST_MODEL)
        print(f"   ⭐ New best IoU: {best_iou:.4f} — saved segformer_best.pth")

print(f"\n✅ Training complete!  Best val IoU: {best_iou:.4f}")

## Cell 9 — Save & Download

In [ ]:
import shutil, zipfile

torch.save({
    "model_state_dict": model.state_dict(),
    "epoch":    NUM_EPOCHS,
    "val_iou":  val_iou,
    "variant":  VARIANT,
    "img_size": IMG_SIZE,
}, FINAL_MODEL)
print(f"💾 Final model → {FINAL_MODEL}")
print(f"💾 Best model  → {BEST_MODEL}")

# Zip .pth files
zip_file = os.path.join(OUTPUT_DIR, f"segformer_{VARIANT}_models.zip")
with zipfile.ZipFile(zip_file, "w") as zf:
    for f in [FINAL_MODEL, BEST_MODEL]:
        if os.path.exists(f):
            zf.write(f, os.path.basename(f))
print(f"📦 Zipped      → {zip_file}")

if PLATFORM == "colab":
    from google.colab import files
    files.download(FINAL_MODEL)
    files.download(BEST_MODEL)
    files.download(zip_file)
else:
    print("\n📂 Kaggle: open Output panel → download files.")
    for f in [FINAL_MODEL, BEST_MODEL, zip_file]:
        if os.path.exists(f):
            print(f"   ✅ {os.path.basename(f)}  ({os.path.getsize(f)/1e6:.0f} MB)")

---
## Using the trained model locally

Place `segformer_best.pth` at `models/checkpoints/DeeplabsV3_road_final.pth` (or update `FINAL_MODEL_PATH` in `config.py`).

Then update `config.py` to load SegFormer at inference:
```python
# config.py
BACKBONE = "segformer-b2"   # triggers build_segformer() in run_inference.py
```

Or call directly:
```python
from models.architecture import build_segformer
model = build_segformer(variant="b2")
ckpt  = torch.load("segformer_best.pth", map_location="cpu")
model.load_state_dict(ckpt["model_state_dict"])
model.eval()
```

---
## Other transformer architectures to explore

| Model | Params | Key idea | Best for |
|---|---|---|---|
| **SegFormer-B2/B5** | 25–82 M | Mix Transformer + MLP decoder | Best ease/accuracy trade-off ← this notebook |
| **Swin-T/S + UPerNet** | 28–50 M | Shifted window attention, hierarchical | Multi-scale features, thin roads at various widths |
| **DPT (Dense Prediction Transformer)** | 123 M | Reassembles ViT features at 4 scales | Dense prediction, strong spatial awareness |
| **Mask2Former + Swin-B** | 69 M | Universal segmentation, deformable attention | If you also want instance or panoptic |
| **InternImage** | 30–220 M | Deformable convolution + transformer hybrid | SOTA on ADE20k, efficient per parameter |

**Recommended next after SegFormer:** Swin-T + UPerNet via `mmsegmentation` or the HuggingFace `UperNetForSemanticSegmentation` — hierarchical windows capture road topology at multiple scales better than any single-scale model.